# 01 - EDA and Data Exploration

This notebook explores the engineered IO workload features, workload mix, time-series behavior, and noisy-neighbor patterns.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "io_features.parquet"
df = pd.read_parquet(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Shape:", df.shape)
print("\nDtypes:")
display(df.dtypes.to_frame("dtype"))
print("\nNull counts:")
display(df.isna().sum().to_frame("null_count").sort_values("null_count", ascending=False))

sns.set_theme(style="whitegrid")
px.defaults.template = "plotly_white"

In [ ]:
# Workload distribution
workload_counts = df.groupby("workload_type", as_index=False).size().sort_values("size", ascending=False)
fig = px.bar(
    workload_counts,
    x="workload_type",
    y="size",
    color="workload_type",
    title="Row Count by Workload Type",
    labels={"size": "row_count", "workload_type": "workload_type"},
)
fig.update_layout(showlegend=False, height=450)
fig.show()

# Per-workload box plots for key metrics
metrics = ["total_iops", "avg_latency_us", "total_throughput_mbps", "sequential_ratio", "capacity_used_pct"]
for metric in metrics:
    fig = px.box(
        df,
        x="workload_type",
        y=metric,
        color="workload_type",
        points="outliers",
        title=f"{metric} by Workload Type",
    )
    fig.update_layout(showlegend=False, height=450)
    fig.show()

# Correlation heatmap of numeric features
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr(numeric_only=True)
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, square=False)
plt.title("Numeric Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Time-series plots for one DB_OLTP, one Backup, one AI_Training volume
target_workloads = ["DB_OLTP", "Backup", "AI_Training"]
sample_volumes = []
for workload in target_workloads:
    subset = df[df["workload_type"] == workload].sort_values("timestamp")
    if not subset.empty:
        sample_volumes.append(subset.iloc[0]["volume_id"])

fig = make_subplots(rows=len(sample_volumes), cols=1, shared_xaxes=True, subplot_titles=sample_volumes)
for idx, volume_id in enumerate(sample_volumes, start=1):
    vol_df = df[df["volume_id"] == volume_id].sort_values("timestamp")
    fig.add_trace(
        go.Scatter(x=vol_df["timestamp"], y=vol_df["total_iops"], mode="lines", name=volume_id),
        row=idx, col=1,
    )
    fig.update_yaxes(title_text="total_iops", row=idx, col=1)
fig.update_layout(height=350 * max(1, len(sample_volumes)), title="Total IOPS Over Time for Sample Volumes")
fig.show()

# Capacity growth for 5 volumes from different tiers
tier_candidates = df.drop_duplicates(["tier", "volume_id"]).sort_values(["tier", "volume_id"])
selected_tier_rows = []
seen_tiers = set()
for _, row in tier_candidates.iterrows():
    tier = row.get("tier")
    if tier in seen_tiers:
        continue
    selected_tier_rows.append(row)
    seen_tiers.add(tier)
    if len(selected_tier_rows) >= 5:
        break
selected_tier_volumes = [row["volume_id"] for row in selected_tier_rows]
fig = make_subplots(rows=len(selected_tier_volumes), cols=1, shared_xaxes=True, subplot_titles=selected_tier_volumes)
for idx, volume_id in enumerate(selected_tier_volumes, start=1):
    vol_df = df[df["volume_id"] == volume_id].sort_values("timestamp")
    fig.add_trace(
        go.Scatter(x=vol_df["timestamp"], y=vol_df["capacity_used_pct"], mode="lines", name=volume_id),
        row=idx, col=1,
    )
    fig.update_yaxes(title_text="capacity_used_pct", row=idx, col=1)
fig.update_layout(height=300 * max(1, len(selected_tier_volumes)), title="Capacity Growth Over Time by Tier Samples")
fig.show()

# Noisy-neighbor evidence: two volumes sharing a node
shared_node = None
pair = None
for node_id, grp in df.groupby("node_id"):
    vols = grp["volume_id"].drop_duplicates().tolist()
    if len(vols) >= 2:
        shared_node = node_id
        pair = vols[:2]
        break

if pair is not None:
    fig = go.Figure()
    for volume_id in pair:
        vol_df = df[df["volume_id"] == volume_id].sort_values("timestamp")
        fig.add_trace(go.Scatter(x=vol_df["timestamp"], y=vol_df["avg_latency_us"], mode="lines", name=volume_id))
    fig.update_layout(title=f"Shared Node Latency Spikes on Node {shared_node}", xaxis_title="timestamp", yaxis_title="avg_latency_us")
    fig.show()
else:
    print("No shared-node volume pair found in the current dataset.")

In [ ]:
# Summary statistics grouped by workload type
key_metrics = ["total_iops", "avg_latency_us", "total_throughput_mbps", "sequential_ratio", "capacity_used_pct"]
summary = df.groupby("workload_type")[key_metrics].agg(["count", "mean", "median", "std", "min", "max"])
display(summary)